In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler


In [2]:
import joblib

model = joblib.load("C:\\Users\\krish\\OneDrive\\Desktop\\RedditTrendPredictor\\models\\trend_model.pkl")
vectorizer = joblib.load("C:\\Users\\krish\\OneDrive\\Desktop\\RedditTrendPredictor\\models\\tfidf_vectorizer.pkl")
scaler = joblib.load("C:\\Users\\krish\\OneDrive\\Desktop\\RedditTrendPredictor\\models\\scaler.pkl")

In [ ]:
import praw
import pandas as pd
from praw.models import MoreComments
from textblob import TextBlob

reddit = praw.Reddit(
    client_id="REDDIT_CLIENT_ID",
    client_secret="REDDIT_CLIENT_SECRET",
    user_agent="REDDIT_USER_AGENT"
)


In [ ]:
def fetch_reddit_data(subreddit_name="technology", limit=800):
    subreddit = reddit.subreddit(subreddit_name)
    posts, comments = [], []

    for post in subreddit.hot(limit=limit):
        posts.append({
            "post_id": post.id,
            "title": post.title,
            "selftext": post.selftext,
            "created_utc": post.created_utc
        })

        post.comments.replace_more(limit=0)
        for c in post.comments.list():
            if isinstance(c, MoreComments):
                continue
            comments.append({
                "post_id": post.id,
                "body": c.body,
                "score": c.score
            })

    return pd.DataFrame(posts), pd.DataFrame(comments)


In [5]:
print(list(scaler.feature_names_in_))


['title_len', 'selftext_len', 'hour', 'dayofweek', 'month', 'sentiment', 'num_comments', 'avg_comment_sentiment', 'avg_comment_score']


In [6]:
def process_comments(df_comments):
    df_comments["sentiment"] = df_comments["body"].apply(
        lambda x: TextBlob(str(x)).sentiment.polarity
    )

    return df_comments.groupby("post_id").agg(
        avg_comment_score=("score", "mean"),
        avg_comment_sentiment=("sentiment", "mean"),
        num_comments=("body", "count")
    ).reset_index()


In [7]:
NUMERIC_FEATURES = list(scaler.feature_names_in_)



def build_features(posts, comment_agg):
    df = posts.merge(comment_agg, on="post_id", how="left")
    df.fillna(0, inplace=True)

    df["created_utc"] = pd.to_datetime(df["created_utc"], unit="s")
    df["hour"] = df["created_utc"].dt.hour
    df["dayofweek"] = df["created_utc"].dt.dayofweek
    df["month"] = df["created_utc"].dt.month

    df["title_len"] = df["title"].astype(str).apply(len)
    df["selftext_len"] = df["selftext"].astype(str).apply(len)
    df["sentiment"] = df["title"].apply(
        lambda x: TextBlob(str(x)).sentiment.polarity
    )

    df["text"] = (df["title"] + " " + df["selftext"]).str.lower()
    return df


In [8]:
print("model:", type(model))
print("vectorizer:", type(vectorizer))
print("scaler:", type(scaler))


model: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
vectorizer: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
scaler: <class 'sklearn.preprocessing._data.StandardScaler'>


In [9]:
from scipy.sparse import hstack

def predict_trending(df):
    X_text = vectorizer.transform(df["text"])
    X_num = scaler.transform(df[NUMERIC_FEATURES])

    X = hstack([X_num, X_text])
    df["trend_probability"] = model.predict_proba(X)[:, 1]

    return df.sort_values("trend_probability", ascending=False)


In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

assert isinstance(model, RandomForestClassifier)
assert isinstance(vectorizer, TfidfVectorizer)
assert isinstance(scaler, StandardScaler)


In [ ]:
import json
from datetime import datetime

# Check if INPUT_DATA was injected (from API) or run standalone
try:
    # If INPUT_DATA exists, use it for single prediction
    if 'INPUT_DATA' in locals() and INPUT_DATA:
        # Map day name to dayofweek number
        day_map = {
            "Monday": 0, "Tuesday": 1, "Wednesday": 2,
            "Thursday": 3, "Friday": 4, "Saturday": 5, "Sunday": 6
        }
        
        # Build a single prediction
        text = (title + " " + selftext).lower()
        sentiment_score = TextBlob(text).sentiment.polarity
        
        numeric_df = pd.DataFrame([{
            "title_len": len(title),
            "selftext_len": len(selftext),
            "hour": hour,
            "dayofweek": day_map.get(dayofweek, 0),
            "month": datetime.now().month,
            "sentiment": sentiment_score,
            "num_comments": 0,
            "avg_comment_sentiment": 0,
            "avg_comment_score": 0
        }])
        
        numeric_df = numeric_df[NUMERIC_FEATURES]
        X_num = scaler.transform(numeric_df)
        X_text = vectorizer.transform([text])
        X = hstack([X_num, X_text])
        
        trend_prob = model.predict_proba(X)[0][1] * 100
        label = "High" if trend_prob >= 70 else "Medium" if trend_prob >= 40 else "Low"
        
        output_result = {
            "trend_probability": round(trend_prob, 2),
            "label": label
        }
        print(json.dumps(output_result))
    else:
        # Fallback: Fetch and predict on Reddit data (standalone mode)
        posts, comments = fetch_reddit_data("technology", limit=200)
        comment_agg = process_comments(comments)
        features = build_features(posts, comment_agg)
        results = predict_trending(features)

        results["trend_probability"] = results["trend_probability"] * 100
        print(results[["title", "trend_probability"]].head(10))

        output_file = "trend_predictions.txt"
        with open(output_file, "w", encoding="utf-8") as f:
            for _, row in results.head(10).iterrows():
                f.write(f"Title: {row['title']}\n")
                f.write(f"Trend Probability: {row['trend_probability']}%\n")
                f.write("-" * 60 + "\n")

        print(f"\nResults saved to {output_file}")
except Exception as e:
    error_result = {
        "trend_probability": 0,
        "label": "Error",
        "error": str(e)
    }
    print(json.dumps(error_result))

                                                 title  trend_probability
50   Cloudflare is laying off 1,100 employees to pr...          68.500000
132  Sony, Nintendo grapple with memory price surge...          64.500000
198  U.S. govt releases highly anticipated files do...          63.500000
163   Android 17 QPR1 Beta 2 Releases for Pixel Phones          63.000000
133  Valve releases Steam Controller CAD files unde...          61.500000
29   Nvidia has already committed $40B to equity AI...          58.500000
28   Fervent And Fake: High-glam AI Avatars Boost T...          57.500000
73                    Cloudflare lays off 1,100 people          56.000000
11   While refrigerators and air conditioners still...          54.416667
172  Nvidia invests $300 million in Corning to buil...          52.916667

Results saved to trend_predictions.txt
